In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API Key is set.")

API Key is set.


In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-nano",temperature=0)

RAG IMPLEMENTATION WITH PDFs

STEP 1: Extracting Text from PDFs

In [4]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader("./docs/fabric-onelake.pdf")

docs = loader.load()

docs[0]

Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creator': 'Microsoft Learn', 'creationdate': '2025-12-12T17:01:10+00:00', 'title': 'fabric onelake | Microsoft Learn', 'moddate': '2025-12-12T17:01:10+00:00', 'source': './docs/fabric-onelake.pdf', 'total_pages': 382, 'page': 0, 'page_label': '1'}, page_content='Tell us about your PDF experience.\nOneLake in Microsoft Fabric\ndocumentation\nOneLake is a single, unified, logical data lake for the whole organization. OneLake comes\nautomatically with every Microsoft Fabric tenant with no infrastructure to manage.\nAbout OneLake\nｅOVERVIEW\nWhat is OneLake?\nOneLake security\nOneLake catalog\nOneLake access and APIs\n｀DEPLOY\nImplement medallion lakehouse architecture\nｂGET STARTED\nCreate a lakehouse with OneLake\nOneLake file explorer\nFind data in the OneLake catalog\nUse Iceberg tables in OneLake\nOneLake shortcuts\nｐCONCEPT\nWhat are shortcuts?\nｂGET STARTED\nCreate a shortcut\nｃHOW-TO GUIDE')

In [6]:
docs[0].metadata

{'producer': 'Microsoft Learn PDF 1.0.25309.01',
 'creator': 'Microsoft Learn',
 'creationdate': '2025-12-12T17:01:10+00:00',
 'title': 'fabric onelake | Microsoft Learn',
 'moddate': '2025-12-12T17:01:10+00:00',
 'source': './docs/fabric-onelake.pdf',
 'total_pages': 382,
 'page': 0,
 'page_label': '1'}

Creating own Metadata for PDF Chunks

In [7]:
for i in docs:

    i.metadata = {"source": "fabric-onelake.pdf",
                  "developer": "Microsoft"}

STEP 2: Splitting the Document into CHUNKS

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100)

chunks = splitter.split_documents(docs)
len(chunks)
chunks[0]

Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content='Tell us about your PDF experience.\nOneLake in Microsoft Fabric\ndocumentation\nOneLake is a single, unified, logical data lake for the whole organization. OneLake comes\nautomatically with every Microsoft Fabric tenant with no infrastructure to manage.\nAbout OneLake\nｅOVERVIEW\nWhat is OneLake?\nOneLake security\nOneLake catalog\nOneLake access and APIs\n｀DEPLOY\nImplement medallion lakehouse architecture\nｂGET STARTED\nCreate a lakehouse with OneLake\nOneLake file explorer\nFind data in the OneLake catalog\nUse Iceberg tables in OneLake\nOneLake shortcuts\nｐCONCEPT\nWhat are shortcuts?\nｂGET STARTED\nCreate a shortcut\nｃHOW-TO GUIDE')

STEP 3: Creating Embeddings for the Chunks

In [10]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

STEP 4: Create and Store Embeddings in Vector Store

In [11]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

STEP 5: Semantic Search

In [12]:
vectorstore.similarity_search("What is Fabric?", k= 3)

[Document(metadata={'developer': 'Microsoft', 'source': 'fabric-onelake.pdf'}, page_content='Discover and explore Fabric items in the OneLake catalog\nView item details\nGovern your data in Fabric\nEndorsement\nFabric domains\nLineage in Fabric\nMonitor hub\n\uf80a\nRelated content'),
 Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content="Fabric capacity and OneLake\nconsumption\nArticle• 12/26/2024\nYou only need one capacity to drive all your Microsoft Fabric experiences, including\nMicrosoft OneLake. Keep reading if you want a detailed example of how OneLake\nconsumes storage and compute.\nOneLake comes automatically with every Fabric tenant and is designed to be the single\nplace for all your analytics data. All the Fabric data items are prewired to store data in\nOneLake. For example, when you store data in a lakehouse or warehouse, your data is\nnatively stored in OneLake.\nWith OneLake, you pay for the data stored, similar to services like A

Talk to LLM

In [15]:
context = vectorstore.similarity_search("What is Fabric?", k= 3)

In [16]:
response = llm.invoke(f"What is Fabric? You can answer using the following context: {context}")
print(response.content)

Microsoft Fabric is a unified data analytics platform from Microsoft. It uses OneLake as the central storage layer and provides a set of analytics experiences (for example, lakehouse and warehouse) that all draw from that same storage.

Key points:
- OneLake comes automatically with every Fabric tenant and serves as the single place for all Fabric data.
- All Fabric data items are stored in OneLake; when you store data in a lakehouse or warehouse, it’s stored there by default.
- You pay for the data stored (like ADLS Gen2 or S3); there are no separate charges for data transactions, which instead consume Fabric capacity that runs your Fabric experiences.
- Fabric includes governance features and a catalog of Fabric items in OneLake, plus capabilities like lineage and a monitor hub.
- It supports integration with external sources (e.g., Snowflake) via OneLake, with setup steps to connect and ingest data.
